# AFL Player Data Cleaning and Validation
This notebook performs data quality assessment, cleaning, validation, and merging of AFL player information and seasonal statistics datasets.
The objective is to prepare a reliable, analysis-ready dataset while documenting every cleaning decision.

In [1]:
import pandas as pd
import numpy as np

In [3]:
players = pd.read_csv("afl_players_info_raw.csv")
stats= pd.read_csv("afl_players_seasonal_stats_raw.csv")

C:\Users\as\AppData\Local\Temp\ipykernel_20432\3610092872.py:2: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  stats= pd.read_csv("afl_players_seasonal_stats_raw.csv")


In [4]:
players.head()
players.info()
players.describe(include='all')
players.shape
players.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2848 entries, 0 to 2847
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   2848 non-null   int64 
 1   player_name          2848 non-null   object
 2   player_full_name     2848 non-null   object
 3   first_name           2848 non-null   object
 4   last_name            2848 non-null   object
 5   born_date            2848 non-null   object
 6   debut_date           2848 non-null   object
 7   debut_age            2848 non-null   int64 
 8   last_date            2848 non-null   object
 9   last_age             2848 non-null   int64 
 10  height               2848 non-null   int64 
 11  weight               2848 non-null   int64 
 12  profile_pic          637 non-null    object
 13  player_link          2848 non-null   object
 14  player_common_names  75 non-null     object
 15  player_teams         2754 non-null   object
dtypes: int

id                        0
player_name               0
player_full_name          0
first_name                0
last_name                 0
born_date                 0
debut_date                0
debut_age                 0
last_date                 0
last_age                  0
height                    0
weight                    0
profile_pic            2211
player_link               0
player_common_names    2773
player_teams             94
dtype: int64

In [6]:
stats.head()
stats.info()
stats.describe(include='all')
stats.shape
stats.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25491 entries, 0 to 25490
Data columns (total 54 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   player_id                    25491 non-null  object 
 1   year                         25491 non-null  int64  
 2   team                         25491 non-null  object 
 3   is_finals                    25491 non-null  bool   
 4   games_played                 25491 non-null  int64  
 5   kicks                        25439 non-null  float64
 6   marks                        25306 non-null  float64
 7   handballs                    25378 non-null  float64
 8   disposals                    25467 non-null  float64
 9   goals                        22954 non-null  float64
 10  behinds                      22802 non-null  float64
 11  hit_outs                     19785 non-null  float64
 12  tackles                      24980 non-null  float64
 13  rebound_50s     

player_id                         0
year                              0
team                              0
is_finals                         0
games_played                      0
kicks                            52
marks                           185
handballs                       113
disposals                        24
goals                          2537
behinds                        2689
hit_outs                       5706
tackles                         511
rebound_50s                    3533
inside_50s                     2971
clearances                     3435
clangers                       2938
free_kicks_for                 1073
free_kicks_against              978
brownlow_votes                 6997
contested_possessions          3336
uncontested_possessions        3322
contested_marks                4925
marks_inside_50                5165
one_percenters                 3643
bounces                        5654
goal_assists                   7092
total_score                 

# Data Quality Assessment

Documenting issues such as
Missing values
Duplicate rows
Incorrect data types
Invalid values
Inconsistent text formatting
Extra spaces
Negative numbers
Impossible ages/heights
Missing IDs
Null player names

# Cleaning Players Dataset

In [17]:
#Remove duplicates
players.drop_duplicates(inplace=True)

In [8]:
#Remove extra spaces
players.columns = players.columns.str.strip()

players["player_name"] = players["player_name"].str.strip()

In [10]:
#Capitalization
players["player_teams"] = players["player_teams"].str.title()

In [11]:
# Convert datatypes
players["height"] = pd.to_numeric(players["height"], errors="coerce")

In [14]:
#Handling missing values
players.dropna(subset=["player_name"], inplace=True)

In [16]:
#Remove impossible values
players = players[players["last_age"] > 15]

players = players[players["height"] > 100]

# Cleaning Seasonal Stats

In [20]:
#Remove Duplicates
original_rows = len(stats)

duplicates = stats.duplicated().sum()
print("Duplicate Rows:", duplicates)

stats.drop_duplicates(inplace=True)

print("Rows after removing duplicates:", len(stats))

Duplicate Rows: 0
Rows after removing duplicates: 25481


In [21]:
#Checking missing values
stats.isnull().sum()

player_id                         0
year                              0
team                              0
is_finals                         0
games_played                      0
kicks                            52
marks                           185
handballs                       113
disposals                        24
goals                          2537
behinds                        2689
hit_outs                       5706
tackles                         511
rebound_50s                    3533
inside_50s                     2971
clearances                     3435
clangers                       2938
free_kicks_for                 1073
free_kicks_against              978
brownlow_votes                 6997
contested_possessions          3336
uncontested_possessions        3322
contested_marks                4925
marks_inside_50                5165
one_percenters                 3643
bounces                        5654
goal_assists                   7092
total_score                 

In [22]:
#Handling missing values for Numeric columns
numeric_cols = stats.select_dtypes(include='number').columns

stats[numeric_cols] = stats[numeric_cols].fillna(0)

In [23]:
#For categorical columns
stats["team"] = stats["team"].fillna("Unknown")

In [24]:
#Remove extra spaces
stats.columns = stats.columns.str.strip()

stats["team"] = stats["team"].str.strip()

In [25]:
#Standardize team names
stats["team"] = stats["team"].str.title()

In [28]:
#Correct data types
stats["year"] = stats["year"].astype(int)

stats["is_finals"] = stats["is_finals"].astype(bool)

In [29]:
#Convert every statistical column to numeric
cols = stats.columns[4:]

stats[cols] = stats[cols].apply(
    pd.to_numeric,
    errors="coerce"
)

In [30]:
#Remove impossible years
stats = stats[
    (stats["year"] >= 1990) &
    (stats["year"] <= 2025)
]

In [31]:
#Remove negative statistics
numeric = stats.select_dtypes(include="number").columns

for col in numeric:
    stats = stats[stats[col] >= 0]

In [32]:
#Validate boolean column
stats["is_finals"].value_counts()

is_finals
False    19356
True      5933
Name: count, dtype: int64

In [33]:
stats["is_finals"] = stats["is_finals"].replace({
    "TRUE":True,
    "FALSE":False
})

In [34]:
#Check duplicate player year records
stats.duplicated(
    subset=["player_id","year","is_finals"]
).sum()

np.int64(0)

In [35]:
#Remove duplicates
stats.drop_duplicates(
    subset=["player_id","year","is_finals"],
    inplace=True
)

In [36]:
#Verify statistical columns
stats.describe()

,year,games_played,kicks,marks,handballs,disposals,goals,behinds,hit_outs,tackles,...,avg_contested_possessions,avg_uncontested_possessions,avg_contested_marks,avg_marks_inside_50,avg_one_percenters,avg_bounces,avg_goal_assists,avg_score,avg_fantasy_points,avg_percentage_played
count,25289.000000,25289.000000,25289.000000,25289.000000,25289.000000,25289.000000,25289.000000,25289.000000,25289.000000,25289.000000,...,25289.000000,25289.000000,25289.000000,25289.000000,25289.000000,25289.000000,25289.000000,25289.000000,25289.000000,25289.000000
mean,2010.347938,10.760251,98.495907,42.619993,67.524813,166.020720,6.433548,4.575823,16.599826,25.604373,...,4.754043,7.514489,0.555910,0.588556,1.815556,0.521583,0.350156,3.579386,59.586702,59.116632
std,9.184119,7.823846,92.248647,39.149830,68.687209,154.241002,10.362801,6.611594,71.747552,26.849594,...,3.011871,4.686843,0.662101,0.792035,1.677483,0.837395,0.467999,4.300423,21.473289,35.422572
min,1990.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2003.000000,3.000000,20.000000,9.000000,13.000000,34.000000,0.000000,0.000000,0.000000,5.000000,...,3.000000,4.400000,0.000000,0.000000,0.800000,0.000000,0.000000,0.400000,45.000000,29.000000
50%,2011.000000,10.000000,66.000000,30.000000,44.000000,114.000000,2.000000,2.000000,0.000000,17.000000,...,4.700000,7.300000,0.300000,0.200000,1.500000,0.100000,0.100000,2.300000,58.800000,76.100000
75%,2018.000000,19.000000,161.000000,70.000000,103.000000,268.000000,8.000000,6.000000,2.000000,37.000000,...,6.300000,10.700000,1.000000,1.000000,2.400000,1.000000,0.500000,5.200000,73.800000,84.300000
max,2025.000000,23.000000,512.000000,226.000000,482.000000,787.000000,123.000000,84.000000,1007.000000,205.000000,...,22.000000,29.000000,6.000000,8.000000,22.000000,13.000000,4.000000,59.000000,149.000000,100.000000


In [37]:
#Final Validation
print("Rows:",len(stats))

print("\nMissing Values")

print(stats.isnull().sum())

print("\nDuplicates")

print(stats.duplicated().sum())

print("\nData Types")

print(stats.dtypes)

Rows: 25289

Missing Values
player_id                      0
year                           0
team                           0
is_finals                      0
games_played                   0
kicks                          0
marks                          0
handballs                      0
disposals                      0
goals                          0
behinds                        0
hit_outs                       0
tackles                        0
rebound_50s                    0
inside_50s                     0
clearances                     0
clangers                       0
free_kicks_for                 0
free_kicks_against             0
brownlow_votes                 0
contested_possessions          0
uncontested_possessions        0
contested_marks                0
marks_inside_50                0
one_percenters                 0
bounces                        0
goal_assists                   0
total_score                    0
total_fantasy_points           0
total_percentag

In [38]:
stats.to_csv(
    "cleaned_seasonal_stats.csv",
    index=False
)

In [39]:
players.isnull().sum()

stats.isnull().sum()

player_id                      0
year                           0
team                           0
is_finals                      0
games_played                   0
kicks                          0
marks                          0
handballs                      0
disposals                      0
goals                          0
behinds                        0
hit_outs                       0
tackles                        0
rebound_50s                    0
inside_50s                     0
clearances                     0
clangers                       0
free_kicks_for                 0
free_kicks_against             0
brownlow_votes                 0
contested_possessions          0
uncontested_possessions        0
contested_marks                0
marks_inside_50                0
one_percenters                 0
bounces                        0
goal_assists                   0
total_score                    0
total_fantasy_points           0
total_percentage_played        0
avg_kicks 

In [40]:
players.duplicated().sum()

stats.duplicated().sum()

np.int64(0)

In [41]:
players.dtypes

stats.dtypes

player_id                       object
year                             int64
team                            object
is_finals                         bool
games_played                     int64
kicks                          float64
marks                          float64
handballs                      float64
disposals                      float64
goals                          float64
behinds                        float64
hit_outs                       float64
tackles                        float64
rebound_50s                    float64
inside_50s                     float64
clearances                     float64
clangers                       float64
free_kicks_for                 float64
free_kicks_against             float64
brownlow_votes                 float64
contested_possessions          float64
uncontested_possessions        float64
contested_marks                float64
marks_inside_50                float64
one_percenters                 float64
bounces                  

In [44]:
players.rename(columns={"id": "player_id"}, inplace=True)

In [46]:
#Merge datasets
merged = pd.merge(
    players,
    stats,
    on="player_id",
    how="left"
)

In [48]:
#Find unmatched IDs
unmatched = stats[
    ~stats["player_id"].isin(players["player_id"])
]

print(unmatched)

      player_id  year                           team  is_finals  games_played  \
239       43292  2003                  Carlton Blues      False             4   
285       43299  2011                St Kilda Saints      False             1   
375       43319  2012  Greater Western Sydney Giants      False             1   
376       43319  2014               Essendon Bombers      False             2   
413       43325  2005                 Brisbane Lions      False             2   
...         ...   ...                            ...        ...           ...   
25486  ID_43738  2013  Greater Western Sydney Giants      False             1   
25487  ID_43660  2015                  Carlton Blues      False            22   
25488  ID_46153  2025               Melbourne Demons      False             5   
25489  ID_44965  2005              Fremantle Dockers      False             6   
25490  ID_45608  1998                  Carlton Blues      False            15   

       kicks  marks  handba

In [49]:
merged[merged.isnull().any(axis=1)]

,player_id,player_name,player_full_name,first_name,last_name,born_date,debut_date,debut_age,last_date,last_age,...,avg_contested_possessions,avg_uncontested_possessions,avg_contested_marks,avg_marks_inside_50,avg_one_percenters,avg_bounces,avg_goal_assists,avg_score,avg_fantasy_points,avg_percentage_played
0,43261,Ryan Abbott,Ryan_Abbott,Ryan,Abbott,1991-06-25,2018-08-02,27,2020-09-05,29,...,6.7,6.7,0.3,1.7,4.3,0.0,0.3,6.7,92.7,84.3
1,43261,Ryan Abbott,Ryan_Abbott,Ryan,Abbott,1991-06-25,2018-08-02,27,2020-09-05,29,...,2.0,3.0,1.0,1.0,5.0,0.0,0.0,1.0,61.0,81.0
2,43261,Ryan Abbott,Ryan_Abbott,Ryan,Abbott,1991-06-25,2018-08-02,27,2020-09-05,29,...,6.0,8.0,0.0,1.0,3.0,0.0,0.0,6.0,74.0,83.0
3,43261,Ryan Abbott,Ryan_Abbott,Ryan,Abbott,1991-06-25,2018-08-02,27,2020-09-05,29,...,0.0,3.0,0.0,1.0,0.0,0.0,0.0,6.0,22.0,54.0
4,43262,Gary Ablett,Gary_Ablett1,Gary,Ablett,1984-05-14,2002-03-30,17,2020-10-24,36,...,5.7,3.0,0.3,0.3,1.0,0.2,0.0,5.3,36.9,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16983,44352,Matt Maguire,Matt_Maguire,matt,MAGUIRE,1984-05-30,2002-05-03,17,2015-04-18,30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16984,45045,Cam Sutcliffe,Cam_Sutcliffe,cam,SUTCLIFFE,1992-05-23,2012-07-08,20,2020-09-05,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16985,45738,Mark McVeigh,Mark_McVeigh,mark,MCVEIGH,1981-01-26,1999-03-25,18,2012-05-26,31,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16986,44919,Dylan Shiel,Dylan_Shiel,dylan,SHIEL,1993-03-09,2012-03-23,19,2025-08-27,32,...,8.3,11.3,0.2,0.2,0.7,0.8,0.5,3.7,72.0,79.2


In [50]:
players.to_csv(
    "cleaned_players_info.csv",
    index=False
)

stats.to_csv(
    "cleaned_seasonal_stats.csv",
    index=False
)

merged.to_csv(
    "merged_players.csv",
    index=False
)

# Validation Report

In [54]:
print("===============================================")
print("           VALIDATION REPORT")
print("================================================")

# Current Row Counts
print("\n1. DATASET SIZE")
print(f"Players Dataset Rows: {len(players)}")
print(f"Seasonal Stats Dataset Rows: {len(stats)}")
print(f"Merged Dataset Rows: {len(merged)}")
# Missing Values
print("\n2. MISSING VALUES REMAINING")

print("\nPlayers Dataset")
print(players.isnull().sum())

print("\nSeasonal Stats Dataset")
print(stats.isnull().sum())
print("\n3. DUPLICATE RECORDS")
print(f"Duplicate rows remaining in Players Dataset: {players.duplicated().sum()}")
print(f"Duplicate rows remaining in Seasonal Stats Dataset: {stats.duplicated().sum()}")


# Data Types
print("\n4. DATA TYPES")

print("\nPlayers Dataset")
print(players.dtypes)

print("\nSeasonal Stats Dataset")
print(stats.dtypes)


# Merge Validation
print("\n5. MERGE VALIDATION")

unmatched = stats.loc[
    ~stats["player_id"].isin(players["player_id"]),
    "player_id"
].unique()

print(f"Unique Players in Merged Dataset: {merged['player_id'].nunique()}")
print(f"Unmatched player_id values: {len(unmatched)}")

if len(unmatched) > 0:
    print("Unmatched IDs:")
    print(unmatched)
else:
    print("All player_id values matched successfully.")

print("\nValidation Completed Successfully!")

           VALIDATION REPORT

1. DATASET SIZE
Players Dataset Rows: 2843
Seasonal Stats Dataset Rows: 25289
Merged Dataset Rows: 16988

2. MISSING VALUES REMAINING

Players Dataset
player_id                 0
player_name               0
player_full_name          0
first_name                0
last_name                 0
born_date                 0
debut_date                0
debut_age                 0
last_date                 0
last_age                  0
height                    0
weight                    0
profile_pic            2207
player_link               0
player_common_names    2768
player_teams             93
dtype: int64

Seasonal Stats Dataset
player_id                      0
year                           0
team                           0
is_finals                      0
games_played                   0
kicks                          0
marks                          0
handballs                      0
disposals                      0
goals                          0
behi